# TP 2 - Clasificacion de paneles solares

Notebook unico para ejecutar y comparar EfficientNetB0 y MobileNetV2.
La grilla usa los tres splits por fuente y registra resultados por arquitectura,
estrategia y split.

**Estrategias:** `random` = pesos aleatorios, `head` = ImageNet con extractor
congelado, `fine_tune` = cabeza y luego extractor con learning rate bajo.


## 1. Preparacion

En Colab, clonar el repositorio o subirlo y ejecutar esta celda desde su raiz.
Los resultados se pueden guardar en una carpeta de Drive para reanudar si se
interrumpe la sesion.

In [ ]:
!pip install -q -r CLASIFICACION/requirements.txt
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path('.').resolve() / 'CLASIFICACION'))

from solar_classification.config import Config, default_dataset_dir, default_results_dir
from solar_classification.engine import verify_setup, run_experiments

DATASET_DIR = default_dataset_dir()
RESULTS_DIR = default_results_dir()
config = Config(DATASET_DIR, RESULTS_DIR)
print('Dataset:', DATASET_DIR)
print('Resultados:', RESULTS_DIR)

## 2. Seleccion de experimentos

Para una prueba corta se puede elegir un solo split y pocos modelos. Para el
resultado final, dejar la grilla completa. El entrenamiento esta pensado para GPU.

In [ ]:
ARCHITECTURES = ['efficientnetb0', 'mobilenetv2']
STRATEGIES = ['random', 'head', 'fine_tune']
SPLITS = ['solarpanels_val', 'geo_kingston_val', 'PV01_val']

# Para un smoke test, por ejemplo: ARCHITECTURES = ['mobilenetv2']; SPLITS = ['PV01_val']
verify_setup(config, ARCHITECTURES, STRATEGIES, SPLITS)

## 3. Entrenamiento y evaluacion

Cada corrida guarda `best.keras`, `metrics.json` e `history.json` dentro de
`CLASIFICACION/results/<arquitectura>_<estrategia>_<split>/`.
El archivo global `metrics.csv` permite comparar todo al final.

In [ ]:
outcome = run_experiments(
    config,
    architectures=ARCHITECTURES,
    strategies=STRATEGIES,
    splits=SPLITS,
)
print('Experimentos completados:', len(outcome['jobs']))

## 4. Comparacion global

La primera tabla resume el promedio entre splits. No reemplaza mirar cada split:
el promedio muestra la tendencia general y la tabla siguiente muestra si el
rendimiento cambia cuando la fuente queda fuera del entrenamiento.

In [ ]:
metrics_path = RESULTS_DIR / 'metrics.csv'
results = pd.read_csv(metrics_path)
display(results.sort_values(['architecture', 'strategy', 'split']))

summary = (results.groupby(['architecture', 'strategy'])
           [['accuracy', 'precision', 'recall', 'f1']]
           .mean().round(3).reset_index())
display(summary.sort_values('f1', ascending=False))

## 5. Graficos para el informe

F1 es especialmente importante por el desbalance. Precision muestra cuantos
positivos predichos son correctos; recall muestra cuantos paneles reales detecta
el clasificador.

In [ ]:
plot_data = results.copy()
plot_data['modelo'] = plot_data['architecture'] + ' / ' + plot_data['strategy']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_data.pivot(index='split', columns='modelo', values='f1').plot(kind='bar', ax=axes[0])
axes[0].set_title('F1 por fuente dejada para validacion')
axes[0].set_ylabel('F1')
axes[0].tick_params(axis='x', rotation=0)
plot_data.groupby('modelo')[['accuracy', 'precision', 'recall', 'f1']].mean().plot(kind='bar', ax=axes[1])
axes[1].set_title('Promedio de metricas')
axes[1].set_ylabel('valor')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
+    plt.show()

## 6. Lectura de resultados

Para la presentacion, responder:

1. ¿ImageNet mejora respecto de `random`?
2. ¿Fine-tuning mejora respecto de entrenar solo la cabeza?
3. ¿Que modelo conserva mejor F1 al cambiar la fuente?
4. ¿Hay un split donde accuracy sea alta pero recall o F1 sean bajos?
5. ¿Las matrices de confusion muestran mas falsos positivos o falsos negativos?

Las conclusiones deben mencionar el split y la metrica concreta, no solo el
promedio global.